Scrivi jobs/01_bronze_ingest.py che:

- definisce lo schema di orders.jsonl con StructType
- legge il JSON con quello schema esplicito
- aggiunge una colonna ingestion_ts (timestamp corrente) e una colonna source_file
- scrive il risultato come Parquet in data/bronze/orders, partizionato per una colonna sensata (pensaci: quale colonna useresti per partizionare dati che cresceranno ogni giorno, e perché?)

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (SparkSession.builder
            .appName("shopflow-hello")
            .master("local[*]")
            .getOrCreate())

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/workspaces/spark")

In [ ]:
raw = spark.read.text (str(PROJECT_ROOT / "data/raw/orders.jsonl"))
raw.show(5, truncate=False)
raw.count()

In [ ]:
orders = spark.read.json(str(PROJECT_ROOT / "data/raw/orders.jsonl"))
orders.printSchema()
orders.show(5, truncate=False)

definiamo lo schema di orders.jsonl con StructType

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType, 
    IntegerType, 
    DoubleType, 
    TimestampType
)

In [ ]:
orders_schema = StructType([
    StructField("channel", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("order_ts", TimestampType(), False),
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True)
])

leggiamo il JSON con quello schema esplicito

In [ ]:
orders = spark.read.schema(orders_schema).json(str(PROJECT_ROOT / "data/raw/orders.jsonl"))
orders.printSchema()
orders.show(5, truncate=False)

aggiungiamo una colonna ingestion_ts (timestamp corrente) e una colonna source_file


In [ ]:
from pyspark.sql import functions as F

orders = (
    orders
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

In [ ]:
orders.show(5, truncate=False)

scriviamo il risultato come Parquet in data/bronze/orders, partizionato per una colonna sensata (pensaci: quale colonna useresti per partizionare dati che cresceranno ogni giorno, e perché?)

In [ ]:
orders_bronze = (
    orders
    .withColumn("ingestion_date", F.to_date("ingestion_ts"))
)

orders_bronze.show(5, truncate=False)

set job group o job description per monitorare meglio il job in Spark UI

In [ ]:
spark.sparkContext.setJobGroup(
"bronze-orders-write",
"RAW orders -> Bronze Parquet"
)

spark.sparkContext.setJobGroup(None, None)


In [ ]:
spark.sparkContext.setJobDescription("WRITE Bronze orders")

scrittura file parquet con modalità overwrite e partizionamento dinamico (per evitare di cancellare partizioni non modificate)

applichiamo coalesce(1) per scrivere un solo file Parquet per partizione (utile per piccoli dataset, ma non scalabile per grandi dataset)

In [ ]:
(
    orders_bronze
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(PROJECT_ROOT / "data/bronze/orders"))

)

generiamo file .py tramite nbconvert:  
jupyter nbconvert --to script ./jobs/01_bronze_ingest.ipynb --output-dir="./scripts"

catch-all - aggiungiamo colonna raw_json per salvare in una colonna del df il JSON originale come stringa (utile per audit o per future elaborazioni)

In [ ]:
raw = spark.read.text(str(PROJECT_ROOT / "data/raw/orders.jsonl")).withColumnRenamed("value", "raw_json")

parsed = raw.withColumn("data", F.from_json("raw_json", orders_schema))

orders_bronze = (
    parsed
    .select("data.*", "raw_json")     # UN SOLO select, entrambe le fonti insieme
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

orders.bronze.show(5, truncate=False)

In [ ]:
spark.stop()